# Messages Analyst
Loads decoded messages and runs analysis.

In [84]:
%run messages_decoder.ipynb

In [85]:
data = decode_messages()

Level-2 OK — 44422 messages, 37 participants.
Level-1 OK — 37 names restored.


In [86]:
DISPLAY_WHITELIST: set[str] = {
    "Duy Lê",
    "Hồng Nhung",
    "Huy Nguyễn",
    "Tri Phan",
    "Linh Tran Hoang",
    "Nhung Tran",
    "Hoàng Hữu Phong",
    "Đinh Thị Nghĩa",
    "Khổng Vũ Minh Thái",
    "Tu Dao Pham",
    "Lê Văn Hữu Thịnh",
    "Jun Ng",
    "Mau Dinh Nguyen",
    "Thắng Quốc Phạm",
    "Tử Kỳ",
    "Dong Mai Hoa",
    "Himiko Tnk",
    "Huỳnh Khang Ninh",
    "May Ca",
    "Bee",
    "Phương Hạ",
    "Lan Hương",
    "Hoàii Thu",
    "Anh Thư",
    "Bế Minh Nhật",
    "Đặng Anh Vũ",
}

def display_name(name: str) -> str:
    if name.startswith('gAAAAA') or name in DISPLAY_WHITELIST:
        return name
    return '***' # + name for debugging

---
## Message count per person
Sorted high → low, with cumulative % of total.

In [87]:
from collections import Counter

counts = Counter(
    msg['sender_name']
    for msg in data['messages']
    if 'sender_name' in msg
)

total = sum(counts.values())
ranked = counts.most_common()

output = []

output.append("## Message count per person")
output.append(f"{'Rank':<5} {'Name':<35} {'Messages':>9} {'Share':>7} {'Cumulative':>11}")
output.append('-' * 72)

cumulative = 0
for rank, (name, count) in enumerate(ranked, start=1):
    share = count / total * 100
    cumulative += share
    output.append(f"{rank:<5} {display_name(name):<35} {count:>9,} {share:>6.1f}% {cumulative:>10.1f}%")

---
## Word count per person
Total words sent by each person (split on spaces), sorted high → low.

In [ ]:
word_counts: Counter = Counter()
for msg in data['messages']:
    name = msg.get('sender_name')
    content = msg.get('content', '')
    if name and content:
        word_counts[name] += len(content.split())

total_words = sum(word_counts.values())
ranked_words = word_counts.most_common()

output.append("\n## Word count per person")
output.append(f"{'Rank':<5} {'Name':<35} {'Words':>10} {'Share':>7} {'Cumulative':>11}")
output.append('-' * 73)

cumulative = 0
for rank, (name, wc) in enumerate(ranked_words, start=1):
    share = wc / total_words * 100
    cumulative += share
    output.append(f"{rank:<5} {display_name(name):<35} {wc:>10,} {share:>6.1f}% {cumulative:>10.1f}%")

---
## Words per message ratio
Average words per message per person, sorted high → low. Only includes people with ≥ 10 messages.

In [ ]:
MIN_MESSAGES_WPM = 10

wpm_data = [
    (name, word_counts.get(name, 0), counts.get(name, 0),
     word_counts.get(name, 0) / counts[name])
    for name in counts
    if counts[name] >= MIN_MESSAGES_WPM and word_counts.get(name, 0) > 0
]
wpm_data.sort(key=lambda x: x[3], reverse=True)

output.append("\n## Words per message ratio")
output.append(f"{'Rank':<5} {'Name':<35} {'Words':>10} {'Messages':>9} {'W/Msg':>7}")
output.append('-' * 71)

for rank, (name, wc, msgs, ratio) in enumerate(wpm_data, start=1):
    output.append(f"{rank:<5} {display_name(name):<35} {wc:>10,} {msgs:>9,} {ratio:>7.2f}")

---
## Reactions given per person
How many reactions each person has given, sorted high → low.

In [88]:
reactions_given = Counter(
    react['actor']
    for msg in data['messages']
    for react in msg.get('reactions', [])
    if 'actor' in react
)

total_given = sum(reactions_given.values())
ranked_given = reactions_given.most_common()

output.append("\n## Reactions given per person")
output.append(f"{'Rank':<5} {'Name':<35} {'Given':>7} {'Share':>7} {'Cumulative':>11}")
output.append('-' * 70)

cumulative = 0
for rank, (name, count) in enumerate(ranked_given, start=1):
    share = count / total_given * 100
    cumulative += share
    output.append(f"{rank:<5} {display_name(name):<35} {count:>7,} {share:>6.1f}% {cumulative:>10.1f}%")

---
## Reactions received per person
How many reactions each person's messages have received, sorted high → low.

In [89]:
reactions_received = Counter(
    msg['sender_name']
    for msg in data['messages']
    if 'sender_name' in msg
    for _ in msg.get('reactions', [])
)

total_received = sum(reactions_received.values())
ranked_received = reactions_received.most_common()

output.append("\n## Reactions received per person")
output.append(f"{'Rank':<5} {'Name':<35} {'Received':>9} {'Share':>7} {'Cumulative':>11}")
output.append('-' * 72)

cumulative = 0
for rank, (name, count) in enumerate(ranked_received, start=1):
    share = count / total_received * 100
    cumulative += share
    output.append(f"{rank:<5} {display_name(name):<35} {count:>9,} {share:>6.1f}% {cumulative:>10.1f}%")

---
## Reactions received / messages sent ratio
Higher ratio = messages tend to generate more reactions. Only includes people with ≥ 10 messages.

In [90]:
MIN_MESSAGES = 10

all_names = set(counts.keys()) | set(reactions_received.keys())
ratios = [
    (name, reactions_received.get(name, 0), counts.get(name, 0),
     reactions_received.get(name, 0) / counts[name])
    for name in all_names
    if counts.get(name, 0) >= MIN_MESSAGES
]
ratios.sort(key=lambda x: x[3], reverse=True)

output.append("\n## Reactions received / messages sent ratio")
output.append(f"{'Rank':<5} {'Name':<35} {'Received':>9} {'Messages':>9} {'Ratio':>7}")
output.append('-' * 70)

for rank, (name, received, msgs, ratio) in enumerate(ratios, start=1):
    output.append(f"{rank:<5} {display_name(name):<35} {received:>9,} {msgs:>9,} {ratio:>7.3f}")

---
## Activity by day of week and time of day (JST)
Setup: convert timestamps to JST and build per-person and group-wide counters.

In [91]:
from datetime import datetime, timezone, timedelta
from collections import defaultdict

JST = timezone(timedelta(hours=9))
DAYS   = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
CHUNKS = ['00-06', '06-12', '12-18', '18-24']
HOURS  = [f'{h:02d}' for h in range(24)]

person_dow   = defaultdict(lambda: [0]*7)
person_chunk = defaultdict(lambda: [0]*4)
person_hour  = defaultdict(lambda: [0]*24)
all_dow   = [0]*7
all_chunk = [0]*4
all_hour  = [0]*24

for msg in data['messages']:
    name = msg.get('sender_name')
    if not name:
        continue
    ts = msg.get('timestamp_ms')
    if ts is None:
        continue
    dt = datetime.fromtimestamp(ts / 1000, tz=JST)
    person_dow[name][dt.weekday()] += 1
    person_chunk[name][dt.hour // 6] += 1
    person_hour[name][dt.hour] += 1
    all_dow[dt.weekday()] += 1
    all_chunk[dt.hour // 6] += 1
    all_hour[dt.hour] += 1

---
## Messages by day of week per person (JST)
Each person’s distribution across Mon–Sun. Counts + share of that person’s total.

In [92]:
COL = 11  # count(4) + space(1) + pct(5) + %(1)

output.append("\n## Messages by day of week per person (JST)")
hdr = f"{'Name':<35}|" + "|".join(f"{d:^{COL}}" for d in DAYS) + "|"
sep = "-" * 35 + "+" + (("-" * COL + "+") * len(DAYS))
output.append(hdr)
output.append(sep)

for name, _ in ranked:
    row = person_dow[name]
    total_p = sum(row)
    if total_p == 0:
        continue
    cols = "|".join(f"{c:>4} {c/total_p*100:>5.1f}%" for c in row)
    output.append(f"{display_name(name):<35}|{cols}|")

---
## Messages by time of day per person (JST, 6-hour chunks)
Each person’s distribution across 00–06, 06–12, 12–18, 18–24.

In [93]:
COL2 = 13  # count(5) + space(1) + pct(6) + %(1)

output.append("\n## Messages by time of day per person (JST, 6h chunks)")
hdr2 = f"{'Name':<35}|" + "|".join(f"{ch:^{COL2}}" for ch in CHUNKS) + "|"
sep2 = "-" * 35 + "+" + (("-" * COL2 + "+") * len(CHUNKS))
output.append(hdr2)
output.append(sep2)

for name, _ in ranked:
    row = person_chunk[name]
    total_p = sum(row)
    if total_p == 0:
        continue
    cols = "|".join(f"{c:>5} {c/total_p*100:>6.1f}%" for c in row)
    output.append(f"{display_name(name):<35}|{cols}|")

---
## Messages by hour per person (JST, 1h)
Each person's distribution across 00–23.

In [94]:
COL3 = 9  # count(3) + space(1) + pct(4) + %(1)

output.append("\n## Messages by hour per person (JST, 1h)")
hdr3 = f"{'Name':<35}|" + "|".join(f"{h:^{COL3}}" for h in HOURS) + "|"
sep3 = "-" * 35 + "+" + (("-" * COL3 + "+") * len(HOURS))
output.append(hdr3)
output.append(sep3)

for name, _ in ranked:
    row = person_hour[name]
    total_p = sum(row)
    if total_p == 0:
        continue
    cols = "|".join(f"{c:>3} {c/total_p*100:>4.0f}%" for c in row)
    output.append(f"{display_name(name):<35}|{cols}|")

---
## Messages by day of week — all (JST)
Group-wide distribution across Mon–Sun.

In [95]:
total_all_dow = sum(all_dow)
output.append("\n## Messages by day of week — all (JST)")
output.append(f"| {'Day':<5}| {'Count':>9} | {'Share':>6} | {'Cumulative':>10} |")
output.append("|-------|-----------|--------|------------|")

cumulative = 0
for d, count in zip(DAYS, all_dow):
    share = count / total_all_dow * 100
    cumulative += share
    output.append(f"| {d:<5}| {count:>9,} | {share:>5.1f}% | {cumulative:>9.1f}% |")

---
## Messages by time of day — all (JST, 6-hour chunks)
Group-wide distribution across 00–06, 06–12, 12–18, 18–24.

In [96]:
total_all_chunk = sum(all_chunk)
output.append("\n## Messages by time of day — all (JST, 6h chunks)")
output.append(f"| {'Chunk':<7}| {'Count':>9} | {'Share':>6} | {'Cumulative':>10} |")
output.append("|---------|-----------|--------|------------|")

cumulative = 0
for ch, count in zip(CHUNKS, all_chunk):
    share = count / total_all_chunk * 100
    cumulative += share
    output.append(f"| {ch:<7}| {count:>9,} | {share:>5.1f}% | {cumulative:>9.1f}% |")

In [97]:
import os

out_path = os.path.normpath(os.path.join(os.getcwd(), '..', 'Data', 'Result', 'analysis.txt'))
os.makedirs(os.path.dirname(out_path), exist_ok=True)

with open(out_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(output))

print(f"Written to {out_path}")

Written to /Users/vudang/Documents/Projects/PhilosophyGroup/messagesAnalysis/Data/Result/analysis.txt
